In [25]:
import numpy as np
import pandas as pd

In [26]:
from surprise import SVD, Dataset, Reader, dump

In [27]:
path = "interactions_data/"

In [28]:
df_movies = pd.read_csv(path + 'movies_data.csv')
df_ratings = pd.read_csv(path + 'interactions_data.csv')

In [29]:
df_movies.rename(columns={'id': 'movie_id', 'title': 'title', 'genres': 'genres'}, inplace=True)

In [30]:
df_movies.head()

,movie_id,title,description,genres,actors,directors
0,37702,Forbidden City Cop,An imperial agent gets ridiculed for his vario...,"Adventure, Action, Comedy","Stephen Chow, Carina Lau, Carman Lee Yeuk-Tung...",Vincent Kok Tak-Chiu
1,1290821,Shelter,A man living in self-imposed exile on a remote...,"Action, Thriller, Crime","Jason Statham, Bodhi Rae Breathnach, Bill Nigh...",Ric Roman Waugh
2,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"Thriller, Crime","Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",Bart Layton
3,1198994,Send Help,Two colleagues become stranded on a deserted i...,"Horror, Comedy, Thriller","Rachel McAdams, Dylan O'Brien, Edyll Ismail, D...",Sam Raimi
4,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"Adventure, Thriller, Science Fiction","Gerard Butler, Morena Baccarin, Roman Griffin ...",Ric Roman Waugh


In [40]:
df_movies.shape

(11440, 6)

In [31]:
df_ratings.head()

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748


In [34]:
algorithm = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[["user_id", "movie_id", "rating"]], reader)
trainset = data.build_full_trainset()
algorithm.fit(trainset)

In [37]:
algorithm.predict(12, 155)


Prediction(uid=12, iid=155, r_ui=None, est=5.0, details={'was_impossible': False})

In [43]:
def recommendation(user_id, num_recommendations):
    # Get a list of all movie IDs
    all_movie_ids = set(df_movies['movie_id'])
    
    # Get the movies the user has already rated
    rated_movie_ids = set(df_ratings[df_ratings['user_id'] == user_id]['movie_id'])
    
    # Get the movies the user has not rated
    unrated_movie_ids = all_movie_ids - rated_movie_ids
    
    # Predict ratings for the unrated movies
    predictions = []
    for movie_id in unrated_movie_ids:
        pred = algorithm.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Sort the predictions by estimated rating in descending order
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Get the top N recommendations
    top_recommendations = predictions[:num_recommendations]
    
    # Retrieve movie titles for the recommended movie IDs
    recommended_movies = []
    for movie_id, est_rating in top_recommendations:
        title = df_movies[df_movies['movie_id'] == movie_id]['title'].values[0]
        recommended_movies.append((title, est_rating))
    
    return recommended_movies

In [44]:
recommendations = recommendation(10, 15)
recommendations

[('The Arctic Convoy', 5.0),
 ('Four Rooms', 5.0),
 ('The Cursed Sanctuary X', 5.0),
 ('Star Wars', 5.0),
 ('Finding Nemo', 5.0),
 ('Forrest Gump', 5.0),
 ('American Beauty', 5.0),
 ('Citizen Kane', 5.0),
 ('Dancer in the Dark', 5.0),
 ('The Fifth Element', 5.0),
 ('Metropolis', 5.0),
 ('Pirates of the Caribbean: The Curse of the Black Pearl', 5.0),
 ('Kill Bill: Vol. 1', 5.0),
 ('Jarhead', 5.0),
 ('9 Songs', 5.0)]

In [45]:
# Xuất mô hình SVD để api.py load (đường dẫn tương đối thư mục model/)
import os

out_dir = os.path.join("collaborative-filtering")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "cf_surprise.dump")
dump.dump(out_path, algo=algorithm, verbose=1)
print("Đã lưu:", os.path.abspath(out_path))

The dump has been saved as file collaborative-filtering\cf_surprise.dump
Đã lưu: d:\CINE_FINAL\cine-ml\model\collaborative-filtering\collaborative-filtering\cf_surprise.dump
